# Medical-SAM3 on AMOS22 MRI / Neurite OASIS (Colab GPU)

Runs the same positional-bias evaluation you already ran with Text3DSAM,
but with [Medical-SAM3](https://github.com/AIM-Research-Lab/Medical-SAM3)
(the 10.3GB checkpoint requires an NVIDIA CUDA GPU — this repo's own docs
and code hard-require it, so it does not run on the CPU/16GB-RAM machine
the rest of this project runs on).

**Before running anything:** `Runtime -> Change runtime type -> T4 GPU`.

For every `(subject, organ)` prompt, this notebook:
1. Runs Medical-SAM3's 3D inference with **one organ prompt at a time**
   (`--instance` mode — a clean binary mask per prompt, no cross-class
   argmax competition), matching exactly how the Text3DSAM run was done.
2. Computes Dice against your real ground-truth label volumes.
3. Computes the predicted centroid/bbox/anatomical-side the same way
   `common3d.py` does for ground truth.
4. Saves a `predictions_medsam3.csv` (same schema as
   `predictions_text3dsam.csv`) and the same publication-style
   Original/Ground-Truth/Prediction figures.

**Time/size expectations:** the checkpoint alone is ~10.3GB to download.
Each (subject, organ) case is a real video/memory-attention forward pass
— budget realistically for this to take a while (likely 1–4+ hours for
the full AMOS22 set of 130 cases, more for OASIS's 350). Run the smoke
test cell first. Google Drive is mounted so partial results survive a
disconnect — don't rely only on the final zip-download cell.

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU visible — set Runtime > Change runtime type > T4 GPU, then re-run."
print("CUDA OK:", torch.cuda.get_device_name(0))

## 2. Mount Drive (persist results across disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/medsam3_amos_oasis'
os.makedirs(DRIVE_ROOT, exist_ok=True)
CKPT_CACHE_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
RESULTS_ROOT = os.path.join(DRIVE_ROOT, 'results')
os.makedirs(CKPT_CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)
print('Drive results will persist under:', RESULTS_ROOT)

## 3. Clone Medical-SAM3 and install

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/AIM-Research-Lab/Medical-SAM3.git
%cd /content/Medical-SAM3
# pyproject.toml pins numpy==1.26 exactly, which has no wheel for Colab's
# current Python version -- relax it to use whatever numpy Colab already has.
!sed -i 's/numpy==1.26/numpy>=1.26/' pyproject.toml
!pip install -q -e .
!pip install -q nibabel pandas matplotlib huggingface_hub

## 4. Clone your research repo (AMOS22 MRI + Neurite OASIS volumes/manifests)

In [ ]:
import getpass
import subprocess

# Private repo -- Colab can't clone it anonymously. Paste a GitHub token
# with read-only access to this repo (Settings > Developer settings >
# Personal access tokens > Fine-grained tokens). Uses subprocess (not the
# `!shell` magic) and scrubs the token from any printed output, so it
# never ends up embedded in a cell's saved output/traceback.
_gh_token = getpass.getpass('GitHub token (repo read access): ')
_clone_url = f'https://{_gh_token}@github.com/fatemafaria142/positional-bias-medical-vlms.git'

_result = subprocess.run(
    ['git', 'clone', '--branch', 'faria/3d-segmentation-datasets', '--depth', '1',
     _clone_url, '/content/research_repo'],
    capture_output=True, text=True,
)
_stderr_scrubbed = _result.stderr.replace(_gh_token, '***')
del _gh_token, _clone_url

if _result.returncode != 0:
    print('git clone FAILED (check token scope/expiry and repo/branch name):')
    print(_stderr_scrubbed)
else:
    print('cloned OK')

RESEARCH_REPO = '/content/research_repo'
print('samples:', os.listdir(os.path.join(RESEARCH_REPO, 'samples')))

## 5. Download the Medical-SAM3 3D checkpoint (~10.3GB, cached to Drive)

In [ ]:
from huggingface_hub import hf_hub_download

CKPT_PATH = hf_hub_download(
    repo_id='Chongcong/Medical-SAM3',
    filename='checkpoint_3D_checkpoint8.pt',
    local_dir=CKPT_CACHE_DIR,
)
print('checkpoint at', CKPT_PATH)

## 6. Helpers (same logic as scripts/common3d.py + the Text3DSAM plotting style)

In [ ]:
import csv, json, subprocess, sys, time
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

MEDSAM3_ROOT = Path('/content/Medical-SAM3')
RESEARCH_REPO = Path(RESEARCH_REPO)

_OPPOSITE = {"R": "L", "L": "R", "A": "P", "P": "A", "S": "I", "I": "S"}

def axcodes_for(nifti_path) -> tuple:
    return nib.aff2axcodes(nib.load(str(nifti_path)).affine)

def mask_ground_truth_3d(mask: np.ndarray, axcodes: tuple):
    coords = np.nonzero(mask)
    if len(coords[0]) == 0:
        return None
    centroid = tuple(float(c.mean()) for c in coords)
    bbox = tuple(int(c.min()) for c in coords) + tuple(int(c.max()) for c in coords)
    sides = []
    for axis, (code, centroid_val) in enumerate(zip(axcodes, centroid)):
        size = mask.shape[axis]
        sides.append(code if centroid_val > size / 2 else _OPPOSITE[code])
    return centroid, bbox, tuple(sides)

def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return 1.0 if denom == 0 else float(2.0 * inter / denom)

def struct_slug(name: str) -> str:
    return name.strip().lower().replace(' ', '_').replace('/', '-')

def window(img_slice):
    lo, hi = np.percentile(img_slice, [1, 99])
    if hi <= lo:
        return img_slice
    return np.clip((img_slice - lo) / (hi - lo), 0, 1)

def load_manifest_by_subject(manifest_path):
    subjects = {}
    with open(manifest_path, newline='') as f:
        for row in csv.DictReader(f):
            subjects.setdefault(row['subject_id'], []).append(row)
    return subjects

GT_COLOR = (0.13, 0.70, 0.30)
PRED_COLOR = (0.85, 0.15, 0.15)
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'axes.titlesize': 12, 'axes.titleweight': 'medium',
    'figure.titlesize': 13, 'legend.fontsize': 10,
    'savefig.facecolor': 'white',
})

def save_panel_figure(out_path, prompt_text, dice, img_slice, gt_slice, pred_slice):
    h, w = img_slice.shape
    fig, axes = plt.subplots(1, 3, figsize=(9.6, 9.6 * h / w / 3 + 0.9),
                              gridspec_kw={'wspace': 0.02})
    ax0, ax1, ax2 = axes
    for ax, label in zip(axes, ('Original', 'Ground Truth', 'Prediction')):
        ax.imshow(img_slice, cmap='gray', origin='lower')
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_xlabel(label, fontsize=11, labelpad=6)
    gt_overlay = np.zeros((*gt_slice.shape, 4)); gt_overlay[gt_slice] = (*GT_COLOR, 0.5)
    ax1.imshow(gt_overlay, origin='lower')
    pred_overlay = np.zeros((*pred_slice.shape, 4)); pred_overlay[pred_slice] = (*PRED_COLOR, 0.5)
    ax2.imshow(pred_overlay, origin='lower')
    legend_handles = [
        plt.Line2D([0], [0], marker='s', color='none', markerfacecolor=GT_COLOR, markersize=12, label='Ground Truth'),
        plt.Line2D([0], [0], marker='s', color='none', markerfacecolor=PRED_COLOR, markersize=12, label='Prediction'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02), frameon=False)
    fig.suptitle(f'“{prompt_text}”', fontsize=13, fontweight='bold', y=1.06)
    ax1.set_title(f'Dice = {dice:.3f}', fontsize=12, fontweight='bold',
                  color=PRED_COLOR if dice < 0.3 else '#1a1a1a', pad=8)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.90, bottom=0.16, wspace=0.02)
    fig.savefig(out_path, dpi=200, bbox_inches='tight', pad_inches=0.15)
    plt.close(fig)

print('helpers ready')

## 7. Dataset config — pick what to run

In [ ]:
DATASETS = {
    'amos22_mri': {'tag': 'amos', 'dir': RESEARCH_REPO / 'samples' / 'amos22_mri'},
    'neurite_oasis': {'tag': 'oasis', 'dir': RESEARCH_REPO / 'samples' / 'neurite_oasis'},
}

# Start with AMOS only (130 cases) — add 'neurite_oasis' once that finishes
# and you're comfortable with the runtime (350 cases, much longer).
RUN_DATASETS = ['amos22_mri']

CASE_NPZ_ROOT = Path('/content/case_npz')
CASE_NPZ_ROOT.mkdir(exist_ok=True)

def build_subject_npz(dataset_name):
    """One .npz per subject with just `imgs` (D,H,W) — prompts are passed
    per-run via --prompt, not embedded, so one image copy covers every organ."""
    cfg = DATASETS[dataset_name]
    manifest_path = cfg['dir'] / f'manifest_{dataset_name}.csv'
    subjects = load_manifest_by_subject(manifest_path)
    out_dir = CASE_NPZ_ROOT / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)
    for subject_id, rows in subjects.items():
        out_path = out_dir / f'{subject_id}.npz'
        if out_path.exists():
            continue
        image_path = cfg['dir'] / rows[0]['image_path']
        imgs = nib.load(str(image_path)).get_fdata().astype(np.float32)
        np.savez_compressed(out_path, imgs=imgs)
    print(f'[{dataset_name}] wrote {len(subjects)} subject npz -> {out_dir}')
    return subjects, out_dir

manifests = {}
case_dirs = {}
for ds in RUN_DATASETS:
    subs, out_dir = build_subject_npz(ds)
    manifests[ds] = subs
    case_dirs[ds] = out_dir

## 8. Smoke test — ONE organ, ONE subject before committing to the full batch

In [ ]:
import shutil

smoke_ds = RUN_DATASETS[0]
smoke_subject = next(iter(manifests[smoke_ds]))
smoke_organ = manifests[smoke_ds][smoke_subject][0]['organ']
print('smoke test:', smoke_ds, smoke_subject, smoke_organ)

smoke_in_dir = Path('/content/smoke_in'); smoke_in_dir.mkdir(exist_ok=True)
smoke_out_dir = Path('/content/smoke_out')
if smoke_out_dir.exists():
    shutil.rmtree(smoke_out_dir)
shutil.copy(case_dirs[smoke_ds] / f'{smoke_subject}.npz', smoke_in_dir / f'{smoke_subject}.npz')

t0 = time.time()
!python /content/Medical-SAM3/medical/inference_3d.py \
  --input {smoke_in_dir} \
  --output-dir {smoke_out_dir} \
  --checkpoint {CKPT_PATH} \
  --prompt "1={smoke_organ}" \
  --instance
print('smoke test elapsed:', round(time.time() - t0, 1), 's')

out_npz = list(smoke_out_dir.glob('*.npz'))
print('output files:', out_npz)
if out_npz:
    segs = np.load(out_npz[0])['segs']
    print('segs shape:', segs.shape, 'dtype:', segs.dtype, 'positive voxels:', int(segs.sum()))

**Check before continuing:** did the smoke test finish without error, and does
`segs shape` match the subject's volume shape? If so, note the elapsed time —
multiply it by (subjects x organs) for each dataset to sanity-check the total
runtime before launching the full loop below.

## 9. Full run — per organ, per dataset (writes to Drive incrementally)

In [ ]:
def unique_organs(subjects_dict):
    """organ -> label_id, in first-seen order, from the manifest (label_id
    is assumed consistent for a given organ name across subjects)."""
    organs = {}
    for rows in subjects_dict.values():
        for row in rows:
            organs.setdefault(row['organ'], int(row['label_id']))
    return organs

def run_dataset(dataset_name):
    cfg = DATASETS[dataset_name]
    subjects = manifests[dataset_name]
    in_dir = case_dirs[dataset_name]
    organs = unique_organs(subjects)

    fig_root = Path(RESULTS_ROOT) / f"results_medsam3_{cfg['tag']}"
    fig_root.mkdir(parents=True, exist_ok=True)
    csv_path = Path(RESULTS_ROOT) / f'predictions_medsam3_{dataset_name}.csv'

    fieldnames = ['subject_id', 'split', 'organ', 'label_id', 'prompt_text',
                  'volume_shape', 'n_pred_voxels', 'dice', 'inference_seconds',
                  'pred_centroid_vox_0', 'pred_centroid_vox_1', 'pred_centroid_vox_2',
                  'pred_bbox_vox_0min', 'pred_bbox_vox_1min', 'pred_bbox_vox_2min',
                  'pred_bbox_vox_0max', 'pred_bbox_vox_1max', 'pred_bbox_vox_2max',
                  'pred_side_axis0', 'pred_side_axis1', 'pred_side_axis2',
                  'gt_centroid_vox_0', 'gt_centroid_vox_1', 'gt_centroid_vox_2',
                  'gt_side_axis0', 'gt_side_axis1', 'gt_side_axis2']

    write_header = not csv_path.exists()
    already_done = set()
    if csv_path.exists():
        with open(csv_path, newline='') as f:
            for row in csv.DictReader(f):
                already_done.add((row['subject_id'], row['organ']))

    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()

        for oi, (organ, label_id) in enumerate(organs.items(), 1):
            organ_out_dir = Path('/content/medsam3_out') / dataset_name / struct_slug(organ)
            organ_out_dir.mkdir(parents=True, exist_ok=True)

            pending_subjects = [s for s in subjects if (s, organ) not in already_done]
            if not pending_subjects:
                print(f'[{oi}/{len(organs)}] {organ}: already done, skipping')
                continue

            print(f'[{oi}/{len(organs)}] running organ "{organ}" over {len(pending_subjects)} subject(s)')
            t0 = time.time()
            result = subprocess.run(
                ['python', str(MEDSAM3_ROOT / 'medical' / 'inference_3d.py'),
                 '--input', str(in_dir),
                 '--output-dir', str(organ_out_dir),
                 '--checkpoint', str(CKPT_PATH),
                 '--prompt', f'1={organ}',
                 '--instance',
                 '--continue-on-error'],
                capture_output=True, text=True,
            )
            elapsed_total = time.time() - t0
            if result.returncode != 0:
                print('  !! non-zero exit, stderr tail:')
                print('  ' + '\n  '.join(result.stderr.strip().splitlines()[-15:]))
            print(f'  organ total: {elapsed_total:.1f}s')

            for subject_id in pending_subjects:
                out_npz_path = organ_out_dir / f'{subject_id}.npz'
                if not out_npz_path.exists():
                    print(f'  [warn] missing output for {subject_id}, skipping')
                    continue

                row0 = next(r for r in subjects[subject_id] if r['organ'] == organ)
                label_path = cfg['dir'] / row0['label_path']
                image_path = cfg['dir'] / row0['image_path']
                label_vol = nib.load(str(label_path)).get_fdata().astype(int)
                axcodes = axcodes_for(image_path)

                pred_mask = np.load(out_npz_path)['segs'].astype(bool)
                gt_mask = label_vol == label_id
                dice = dice_score(pred_mask, gt_mask)
                n_voxels = int(pred_mask.sum())
                gt = mask_ground_truth_3d(pred_mask, axcodes) if n_voxels > 0 else None

                out_row = {
                    'subject_id': subject_id, 'split': row0['split'], 'organ': organ,
                    'label_id': label_id, 'prompt_text': row0['prompt_text'],
                    'volume_shape': row0['volume_shape'], 'n_pred_voxels': n_voxels,
                    'dice': round(dice, 4), 'inference_seconds': round(elapsed_total / max(len(pending_subjects), 1), 2),
                    'pred_centroid_vox_0': None, 'pred_centroid_vox_1': None, 'pred_centroid_vox_2': None,
                    'pred_bbox_vox_0min': None, 'pred_bbox_vox_1min': None, 'pred_bbox_vox_2min': None,
                    'pred_bbox_vox_0max': None, 'pred_bbox_vox_1max': None, 'pred_bbox_vox_2max': None,
                    'pred_side_axis0': None, 'pred_side_axis1': None, 'pred_side_axis2': None,
                    'gt_centroid_vox_0': row0['gt_centroid_vox_0'], 'gt_centroid_vox_1': row0['gt_centroid_vox_1'],
                    'gt_centroid_vox_2': row0['gt_centroid_vox_2'], 'gt_side_axis0': row0['gt_side_axis0'],
                    'gt_side_axis1': row0['gt_side_axis1'], 'gt_side_axis2': row0['gt_side_axis2'],
                }
                if gt is not None:
                    (cx, cy, cz), bbox, (s0, s1, s2) = gt
                    out_row.update({
                        'pred_centroid_vox_0': cx, 'pred_centroid_vox_1': cy, 'pred_centroid_vox_2': cz,
                        'pred_bbox_vox_0min': bbox[0], 'pred_bbox_vox_1min': bbox[1], 'pred_bbox_vox_2min': bbox[2],
                        'pred_bbox_vox_0max': bbox[3], 'pred_bbox_vox_1max': bbox[4], 'pred_bbox_vox_2max': bbox[5],
                        'pred_side_axis0': s0, 'pred_side_axis1': s1, 'pred_side_axis2': s2,
                    })
                writer.writerow(out_row); f.flush()
                print(f'    {subject_id:24s} dice={dice:.3f} voxels={n_voxels}')

                image_np = nib.load(str(image_path)).get_fdata().astype(np.float32)
                slice_axis = int(np.argmin(image_np.shape))
                gt_coords = np.nonzero(gt_mask)
                slice_idx = int(round(gt_coords[slice_axis].mean())) if len(gt_coords[0]) else image_np.shape[slice_axis] // 2
                def take_slice(vol, ax=slice_axis, idx=slice_idx):
                    sl = [slice(None)] * 3; sl[ax] = idx
                    return vol[tuple(sl)].T
                subj_fig_dir = fig_root / subject_id
                subj_fig_dir.mkdir(parents=True, exist_ok=True)
                save_panel_figure(
                    subj_fig_dir / f'{struct_slug(organ)}.png',
                    row0['prompt_text'], dice,
                    window(take_slice(image_np)), take_slice(gt_mask), take_slice(pred_mask),
                )

    print(f'[{dataset_name}] done -> {csv_path}, {fig_root}')

for ds in RUN_DATASETS:
    run_dataset(ds)

The loop writes each row to the CSV (with `f.flush()`) as soon as it's
computed and skips already-completed `(subject, organ)` pairs on re-run —
so if Colab disconnects partway through, just re-run this cell (after
re-running cells 1–8 to restore state) and it resumes where it left off,
since results live under `RESULTS_ROOT` on Drive.

## 10. Quick summary + zip for download (optional — results are already on Drive)

In [ ]:
import pandas as pd

for ds in RUN_DATASETS:
    csv_path = Path(RESULTS_ROOT) / f'predictions_medsam3_{ds}.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        print(ds, '-> mean Dice:', round(df['dice'].mean(), 4), f'({len(df)} rows)')
        print(df.groupby('organ')['dice'].mean().sort_values(ascending=False).round(3))
        print()

shutil.make_archive('/content/medsam3_results', 'zip', RESULTS_ROOT)
from google.colab import files
files.download('/content/medsam3_results.zip')

## 11. Pulling results back into your local repo

Download `medsam3_results.zip` above (or grab the files straight from
`My Drive/medsam3_amos_oasis/results/` in Drive), then on your local
machine, unzip it into the repo root so you get:

```
positional-bias-medical-vlms/
  results_medsam3_amos/<subject>/<organ>.png
  results_medsam3_oasis/<subject>/<structure>.png
  samples/amos22_mri/predictions_medsam3.csv        (rename from predictions_medsam3_amos22_mri.csv)
  samples/neurite_oasis/predictions_medsam3.csv     (rename from predictions_medsam3_neurite_oasis.csv)
```

Then it's a normal `git add` / `git commit` / `git push`, same as the
Text3DSAM results — tell your assistant to do this and it can rename/move
the files and push for you once they're unzipped locally.